# Semana 8: Modelos Seq2Seq y Atención
## Notebook Conceptual (NB1) – Traducción con Datos Dummy

**Propósito:** Implementar arquitecturas encoder-decoder con mecanismo de atención para tareas como traducción automática.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Construir un encoder RNN que comprime la frase origen.
- Construir un decoder RNN que genera la frase destino.
- Implementar el mecanismo de atención (Luong) paso a paso.
- Visualizar los pesos de atención para cada paso de generación.
- Entrenar el modelo en un corpus minúsculo y observar las traducciones.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos PyTorch.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configuración de PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\nLibrerías importadas correctamente.")

---
## 1. Datos Dummy: Traducción Inglés-Español

Creamos un pequeño corpus de pares de frases inventadas para traducción.

In [ ]:
# Pares de frases (inglés -> español)
pairs = [
    ("i love cats", "amo a los gatos"),
    ("i love dogs", "amo a los perros"),
    ("the cat is black", "el gato es negro"),
    ("the dog is white", "el perro es blanco"),
    ("i love my cat", "amo a mi gato"),
    ("i love my dog", "amo a mi perro"),
    ("the cat loves milk", "el gato ama la leche"),
    ("the dog loves meat", "el perro ama la carne")
]

print("=== CORPUS DE TRADUCCIÓN ===")
for eng, esp in pairs:
    print(f"{eng:20} -> {esp}")

---
## 2. Construcción de Vocabularios

Creamos vocabularios para inglés (origen) y español (destino), incluyendo tokens especiales.

In [ ]:
def build_vocab(sentences, add_special_tokens=True):
    """Construye vocabulario a partir de una lista de oraciones."""
    word_counts = {}
    for sent in sentences:
        for word in sent.split():
            word_counts[word] = word_counts.get(word, 0) + 1
    
    vocab = {word: idx+4 for idx, word in enumerate(sorted(word_counts.keys()))}
    
    if add_special_tokens:
        vocab['<PAD>'] = 0
        vocab['<SOS>'] = 1
        vocab['<EOS>'] = 2
        vocab['<UNK>'] = 3
    
    return vocab

# Vocabulario inglés
eng_sentences = [pair[0] for pair in pairs]
eng_vocab = build_vocab(eng_sentences)
eng_idx2word = {idx: word for word, idx in eng_vocab.items()}

# Vocabulario español
esp_sentences = [pair[1] for pair in pairs]
esp_vocab = build_vocab(esp_sentences)
esp_idx2word = {idx: word for word, idx in esp_vocab.items()}

print(f"Tamaño vocabulario inglés: {len(eng_vocab)}")
print(f"Tamaño vocabulario español: {len(esp_vocab)}")
print("\nVocabulario inglés:", list(eng_vocab.keys())[:10])
print("Vocabulario español:", list(esp_vocab.keys())[:10])

### 2.1. Codificación de oraciones

In [ ]:
def encode_sentence(sentence, vocab, max_len=10):
    """Convierte una oración a índices con tokens SOS y EOS y padding."""
    tokens = sentence.split()
    indices = [vocab.get(word, vocab['<UNK>']) for word in tokens]
    # Añadir SOS al inicio y EOS al final
    indices = [vocab['<SOS>']] + indices + [vocab['<EOS>']]
    # Padding
    indices += [vocab['<PAD>']] * (max_len + 2 - len(indices))
    return indices[:max_len+2]

max_len = 8  # longitud máxima de palabras en la frase (sin contar SOS/EOS)

# Codificar todos los pares
X = torch.tensor([encode_sentence(p[0], eng_vocab, max_len) for p in pairs])
Y = torch.tensor([encode_sentence(p[1], esp_vocab, max_len) for p in pairs])

print(f"Forma de X: {X.shape}")
print(f"Forma de Y: {Y.shape}")
print("\nEjemplo codificado:")
print(f"Frase inglés: {pairs[0][0]}")
print(f"Índices: {X[0].tolist()}")
print(f"Frase español: {pairs[0][1]}")
print(f"Índices: {Y[0].tolist()}")

---
## 3. Encoder RNN

El encoder procesa la frase origen y devuelve todos los estados ocultos.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch, seq_len, embed_dim)
        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: (batch, seq_len, hidden_dim) - todos los estados ocultos
        # hidden, cell: (num_layers, batch, hidden_dim)
        return outputs, hidden, cell

# Parámetros
EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.3

encoder = Encoder(len(eng_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
print("Encoder creado.")

---
## 4. Decoder con Atención (Luong)

Implementamos el mecanismo de atención de Luong (multiplicativa).

In [ ]:
class DecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.3):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.attn_proj = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, y_prev, hidden, cell, encoder_outputs):
        # y_prev: (batch, 1) - palabra anterior
        # hidden, cell: (num_layers, batch, hidden_dim)
        # encoder_outputs: (batch, seq_len, hidden_dim)
        
        # Embedding de la palabra anterior
        embedded = self.dropout(self.embedding(y_prev))  # (batch, 1, embed_dim)
        
        # Paso por LSTM
        lstm_out, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # lstm_out: (batch, 1, hidden_dim)
        
        # Calcular atención (producto punto)
        # lstm_out: (batch, 1, hidden_dim) -> (batch, hidden_dim)
        query = lstm_out.squeeze(1)  # (batch, hidden_dim)
        
        # encoder_outputs: (batch, seq_len, hidden_dim)
        # scores: (batch, seq_len) = query * encoder_outputs (producto punto)
        scores = torch.bmm(encoder_outputs, query.unsqueeze(2)).squeeze(2)  # (batch, seq_len)
        
        # Normalizar con softmax
        attn_weights = torch.softmax(scores, dim=1)  # (batch, seq_len)
        
        # Vector de contexto ponderado
        # attn_weights: (batch, 1, seq_len), encoder_outputs: (batch, seq_len, hidden_dim)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)  # (batch, hidden_dim)
        
        # Combinar query y contexto
        combined = torch.tanh(self.attn_proj(torch.cat((query, context), dim=1)))  # (batch, hidden_dim)
        
        # Capa de salida
        output = self.fc_out(self.dropout(combined))  # (batch, vocab_size)
        
        return output, hidden, cell, attn_weights

decoder = DecoderWithAttention(len(esp_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
print("Decoder con atención creado.")

---
## 5. Modelo Seq2Seq Completo

Integramos encoder y decoder en un modelo completo con teacher forcing.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: (batch, src_len)
        # trg: (batch, trg_len)
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.vocab_size
        
        # Tensor para guardar las salidas
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        # Encoder
        encoder_outputs, hidden, cell = self.encoder(src)
        
        # Primer input al decoder: token <SOS>
        decoder_input = torch.ones(batch_size, 1).long().to(self.device)  # <SOS> = 1
        
        for t in range(trg_len):
            output, hidden, cell, attn = self.decoder(decoder_input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            
            # Teacher forcing
            if torch.rand(1).item() < teacher_forcing_ratio:
                decoder_input = trg[:, t].unsqueeze(1)  # palabra correcta
            else:
                decoder_input = output.argmax(1).unsqueeze(1)  # palabra predicha
        
        return outputs

model = Seq2Seq(encoder, decoder, device).to(device)
print("Modelo Seq2Seq creado.")

---
## 6. Entrenamiento del Modelo

### 6.1. Preparación de datos

In [ ]:
# Crear DataLoader
dataset = TensorDataset(X, Y)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Función de pérdida (ignorar padding)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 200
losses = []

print("Iniciando entrenamiento...")
for epoch in range(epochs):
    epoch_loss = 0
    for batch_X, batch_Y in dataloader:
        batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_X, batch_Y, teacher_forcing_ratio=0.5)
        
        # Reshape para la pérdida
        loss = criterion(output.reshape(-1, len(esp_vocab)), batch_Y.reshape(-1))
        loss.backward()
        
        # Gradient clipping para evitar explosión
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        
        optimizer.step()
        epoch_loss += loss.item()
    
    losses.append(epoch_loss / len(dataloader))
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {losses[-1]:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.title('Evolución de la pérdida durante entrenamiento')
plt.grid(True)
plt.show()

---
## 7. Traducción y Visualización de Atención

### 7.1. Función de traducción (greedy decoding)

In [ ]:
def translate(model, sentence, eng_vocab, esp_vocab, max_len=10):
    """Traduce una oración del inglés al español."""
    model.eval()
    
    # Codificar oración origen
    indices = encode_sentence(sentence, eng_vocab, max_len)
    src_tensor = torch.tensor([indices]).to(device)
    
    with torch.no_grad():
        # Encoder
        encoder_outputs, hidden, cell = model.encoder(src_tensor)
        
        # Decodificación greedy
        decoder_input = torch.ones(1, 1).long().to(device)  # <SOS>
        translated_indices = []
        attention_weights = []
        
        for _ in range(max_len + 2):  # +2 por SOS/EOS
            output, hidden, cell, attn = model.decoder(decoder_input, hidden, cell, encoder_outputs)
            attention_weights.append(attn.cpu().numpy())
            
            predicted = output.argmax(1).item()
            if predicted == esp_vocab['<EOS>']:
                break
            if predicted != esp_vocab['<PAD>']:
                translated_indices.append(predicted)
            
            decoder_input = torch.tensor([[predicted]]).to(device)
    
    # Convertir índices a palabras
    translated_words = [esp_idx2word[idx] for idx in translated_indices]
    return ' '.join(translated_words), np.array(attention_weights)

In [ ]:
# Probar algunas frases
test_sentences = [
    "i love cats",
    "the dog is white",
    "i love my cat",
    "the cat loves milk"
]

print("=== TRADUCCIONES ===\n")
for sent in test_sentences:
    translation, _ = translate(model, sent, eng_vocab, esp_vocab)
    print(f"Inglés: {sent:20} -> Español: {translation}")

### 7.2. Visualización de pesos de atención

In [ ]:
def plot_attention(sentence, translation, attention_weights, eng_vocab, esp_vocab):
    """Visualiza la matriz de atención."""
    # Obtener palabras origen
    src_words = sentence.split()
    src_words = ['<SOS>'] + src_words + ['<EOS>']
    
    # Palabras destino (sin contar SOS/EOS de la salida)
    trg_words = translation.split()
    
    # Atención tiene forma (trg_len, 1, src_len) - eliminar dimensión 1
    attn_matrix = attention_weights.squeeze(1)[:len(trg_words), :len(src_words)]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(attn_matrix, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=src_words, yticklabels=trg_words)
    plt.xlabel('Palabras origen')
    plt.ylabel('Palabras destino')
    plt.title('Matriz de Atención')
    plt.show()

# Visualizar atención para una frase
test_sent = "i love cats"
translation, attn_weights = translate(model, test_sent, eng_vocab, esp_vocab)
print(f"Frase: {test_sent}")
print(f"Traducción: {translation}")
plot_attention(test_sent, translation, attn_weights, eng_vocab, esp_vocab)

In [ ]:
# Probar con otra frase
test_sent = "the cat loves milk"
translation, attn_weights = translate(model, test_sent, eng_vocab, esp_vocab)
print(f"Frase: {test_sent}")
print(f"Traducción: {translation}")
plot_attention(test_sent, translation, attn_weights, eng_vocab, esp_vocab)

### Interpretación de la matriz de atención:

- Cada fila corresponde a una palabra generada en español.
- Cada columna corresponde a una palabra de la frase original en inglés.
- Los valores indican cuánta atención prestó el decodificador a cada palabra origen al generar la palabra destino.
- Idealmente, vemos una alineación diagonal: "i" atiende a "amo", "love" atiende a "a", "cats" atiende a "gatos", etc.
- Esto demuestra que el modelo ha aprendido a alinear las palabras correctamente.

---
## 8. Ejercicio para el Estudiante

1. **Amplía el corpus**: Añade más pares de frases y observa cómo mejora el modelo.
2. **Experimenta con la atención de Bahdanau**: Implementa la atención aditiva en lugar de la multiplicativa y compara.
3. **Beam Search**: Implementa decoding con beam search en lugar de greedy.
4. **Evalúa con BLEU**: Usa la librería `sacrebleu` para evaluar las traducciones.
5. **Prueba con frases fuera del vocabulario**: Por ejemplo, "i love pizza" (pizza no está en el vocabulario).

In [ ]:
# Espacio para el estudiante
# ...

---
## 9. Conclusiones

En este notebook hemos implementado un modelo Seq2Seq con atención para traducción automática:

✔️ **Encoder**: Procesa la frase origen y devuelve todos los estados ocultos.

✔️ **Decoder con atención (Luong)**: Genera la frase destino, usando atención para enfocarse en partes relevantes de la entrada.

✔️ **Visualización de atención**: Las matrices de atención muestran la alineación aprendida entre palabras origen y destino.

✔️ **Entrenamiento**: Con un corpus pequeño, el modelo aprende a traducir frases similares a las de entrenamiento.

**Lección clave**: El mecanismo de atención es fundamental para manejar dependencias de largo alcance y alinear secuencias de diferente longitud. Es la base de los modelos Transformer que veremos en la próxima sesión.

---
**Fin del Notebook Conceptual - Semana 8 (NLP)**